# Topic 2 — Hands-On: Building a Neural Network
## Understanding Forward Propagation, Activation Functions & Gradient Descent

> **Objective:** In the theory session we learnt *what* a neural network does.  
> Now we will **build one step-by-step** and watch it learn!

### What We Will Cover
1. **Understanding** how a neuron works (weighted sum + activation)  
2. **Visualizing** activation functions  
3. **Building** a simple neural network from scratch  
4. **Training** the network (forward pass → loss → backpropagation → gradient descent)  
5. **Experimenting** — change learning rates, activation functions, and see what happens!

---
## 1. Setup — Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print("Libraries loaded!")

---
## 2. How a Neuron Works

Let's build a single neuron step by step:
1. Take inputs
2. Multiply each by a weight
3. Add them up + add a bias
4. Apply activation function

In [ ]:
# A single neuron
def neuron(inputs, weights, bias):
    """
    Simple neuron: weighted sum + ReLU activation
    """
    # Step 1: Weighted sum
    weighted_sum = np.dot(inputs, weights) + bias
    print(f"  Weighted sum: {weighted_sum:.4f}")
    
    # Step 2: ReLU activation
    output = max(0, weighted_sum)
    print(f"  After ReLU: {output:.4f}")
    
    return output

# Test it!
print("Testing a single neuron:")
print("-" * 40)
inputs = np.array([0.5, 0.8])
weights = np.array([0.3, 0.7])
bias = 0.1

print(f"Inputs: {inputs}")
print(f"Weights: {weights}")
print(f"Bias: {bias}")
print(f"\nNeuron computation:")
result = neuron(inputs, weights, bias)
print(f"\nFinal output: {result}")

---
## 3. Activation Functions

Let's visualize the three main activation functions and see how they differ.

In [ ]:
# Define activation functions
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

# Plot them
x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Activation Functions', fontsize=14, fontweight='bold')

# ReLU
axes[0].plot(x, relu(x), 'r-', linewidth=2)
axes[0].set_title('ReLU', fontsize=12)
axes[0].set_xlabel('Input')
axes[0].set_ylabel('Output')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle='-', alpha=0.3)

# Sigmoid
axes[1].plot(x, sigmoid(x), 'b-', linewidth=2)
axes[1].set_title('Sigmoid', fontsize=12)
axes[1].set_xlabel('Input')
axes[1].set_ylabel('Output')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.1, 1.1)

# Tanh
axes[2].plot(x, tanh(x), 'g-', linewidth=2)
axes[2].set_title('Tanh', fontsize=12)
axes[2].set_xlabel('Input')
axes[2].set_ylabel('Output')
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(-1.1, 1.1)

plt.tight_layout()
plt.show()

# Test with examples
print("\nTesting activation functions:")
print("-" * 40)
test_values = [-3, -1, 0, 1, 3]
print(f"Input: {test_values}")
print(f"ReLU: {[relu(x) for x in test_values]}")
print(f"Sigmoid: {[round(sigmoid(x), 3) for x in test_values]}")
print(f"Tanh: {[round(tanh(x), 3) for x in test_values]}")

---
## 4. Building a Simple Neural Network

Let's build a network that learns to solve a simple problem: **predict if a point is in Class 0 or Class 1**.

In [ ]:
# Generate simple data
np.random.seed(42)

# Two classes of points
class_0 = np.random.randn(50, 2) + np.array([0, 0])  # Centered at (0,0)
class_1 = np.random.randn(50, 2) + np.array([3, 3])  # Centered at (3,3)

X = np.vstack([class_0, class_1])  # All points
y = np.array([0]*50 + [1]*50)       # Labels (0 or 1)

# Plot the data
plt.figure(figsize=(8, 6))
plt.scatter(class_0[:, 0], class_0[:, 1], c='red', label='Class 0', alpha=0.6)
plt.scatter(class_1[:, 0], class_1[:, 1], c='blue', label='Class 1', alpha=0.6)
plt.xlabel('X₁')
plt.ylabel('X₂')
plt.title('Our Simple Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Total points: {len(X)}")
print(f"Class 0: {np.sum(y == 0)} points")
print(f"Class 1: {np.sum(y == 1)} points")

---
## 5. Define the Network

Our network will be:
- **Input layer**: 2 neurons (x₁ and x₂ coordinates)
- **Hidden layer**: 10 neurons with ReLU
- **Output layer**: 1 neuron with Sigmoid (probability of Class 1)

In [ ]:
# Network architecture
input_size = 2    # x₁ and x₂
hidden_size = 10  # neurons in hidden layer
output_size = 1   # probability of class 1

# Initialize weights randomly (small values)
np.random.seed(42)
W1 = np.random.randn(input_size, hidden_size) * 0.01  # input → hidden
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.01  # hidden → output
b2 = np.zeros((1, output_size))

print("Network Architecture:")
print(f"- Input layer: {input_size} neurons")
print(f"- Hidden layer: {hidden_size} neurons (ReLU activation)")
print(f"- Output layer: {output_size} neuron (Sigmoid activation)")
print(f"\nTotal parameters: {input_size*hidden_size + hidden_size + hidden_size*output_size + output_size}")

---
## 6. Training Loop

Now let's train the network! The training loop:
1. **Forward pass**: Data flows through network → get prediction
2. **Compute loss**: Compare prediction to true label
3. **Backward pass**: Calculate gradients
4. **Update weights**: Adjust using gradient descent

In [ ]:
# Training hyperparameters
learning_rate = 0.1
epochs = 200

# Storage for tracking
losses = []
accuracies = []

print("Training the network...")
print("-" * 50)

for epoch in range(epochs):
    # ============ FORWARD PASS ============
    # Layer 1: input → hidden
    z1 = X.dot(W1) + b1
    a1 = relu(z1)  # Apply ReLU
    
    # Layer 2: hidden → output
    z2 = a1.dot(W2) + b2
    a2 = sigmoid(z2)  # Apply Sigmoid
    
    # ============ COMPUTE LOSS ============
    # Binary Cross-Entropy Loss
    epsilon = 1e-8  # Prevent log(0)
    loss = -np.mean(y.reshape(-1, 1) * np.log(a2 + epsilon) + 
                   (1 - y.reshape(-1, 1)) * np.log(1 - a2 + epsilon))
    losses.append(loss)
    
    # Compute accuracy
    predictions = (a2 > 0.5).astype(int).flatten()
    accuracy = np.mean(predictions == y)
    accuracies.append(accuracy)
    
    # ============ BACKWARD PASS ============
    # Compute gradients
    dz2 = a2 - y.reshape(-1, 1)
    dW2 = a1.T.dot(dz2) / len(X)
    db2 = np.mean(dz2, axis=0, keepdims=True)
    
    da1 = dz2.dot(W2.T)
    dz1 = da1 * (z1 > 0)  # ReLU derivative
    dW1 = X.T.dot(dz1) / len(X)
    db1 = np.mean(dz1, axis=0, keepdims=True)
    
    # ============ UPDATE WEIGHTS ============
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    
    # Print progress
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {loss:.4f} | Accuracy: {accuracy:.2%}")

print("\nTraining complete!")

---
## 7. Visualize Training Results

In [ ]:
# Plot training progress
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Loss curve
axes[0].plot(losses, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss (Lower = Better)', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(accuracies, 'g-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy (Higher = Better)', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f"Final accuracy: {accuracies[-1]:.2%}")

---
## 8. Visualize What the Network Learned

Let's see the decision boundary — how the network separates the two classes.

In [ ]:
# Create a mesh grid to visualize decision boundary
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))

# Predict on the mesh grid
grid_points = np.c_[xx.ravel(), yy.ravel()]
z1_grid = grid_points.dot(W1) + b1
a1_grid = relu(z1_grid)
z2_grid = a1_grid.dot(W2) + b2
a2_grid = sigmoid(z2_grid)
Z = a2_grid.reshape(xx.shape)

# Plot
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
plt.colorbar(label='Probability of Class 1')
plt.scatter(class_0[:, 0], class_0[:, 1], c='red', label='Class 0', edgecolors='black')
plt.scatter(class_1[:, 0], class_1[:, 1], c='blue', label='Class 1', edgecolors='black')
plt.xlabel('X₁')
plt.ylabel('X₂')
plt.title('Network Decision Boundary\n(Red = Class 0, Blue = Class 1)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("The network learned to separate the two classes!")
print("The color shows the network's confidence (red = Class 0, blue = Class 1).")

---
## 9. Experiment: Change the Learning Rate

Let's see what happens when we use different learning rates.

In [ ]:
# Train with different learning rates
learning_rates = [0.001, 0.1, 1.0]
colors = ['red', 'green', 'blue']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Effect of Learning Rate', fontsize=14, fontweight='bold')

for idx, lr in enumerate(learning_rates):
    # Reinitialize weights
    np.random.seed(42)
    W1_temp = np.random.randn(input_size, hidden_size) * 0.01
    b1_temp = np.zeros((1, hidden_size))
    W2_temp = np.random.randn(hidden_size, output_size) * 0.01
    b2_temp = np.zeros((1, output_size))
    
    temp_losses = []
    
    for epoch in range(200):
        # Forward
        z1 = X.dot(W1_temp) + b1_temp
        a1 = relu(z1)
        z2 = a1.dot(W2_temp) + b2_temp
        a2 = sigmoid(z2)
        
        # Loss
        loss = -np.mean(y.reshape(-1, 1) * np.log(a2 + 1e-8) + 
                       (1 - y.reshape(-1, 1)) * np.log(1 - a2 + 1e-8))
        temp_losses.append(loss)
        
        # Backward
        dz2 = a2 - y.reshape(-1, 1)
        dW2 = a1.T.dot(dz2) / len(X)
        db2 = np.mean(dz2, axis=0, keepdims=True)
        da1 = dz2.dot(W2_temp.T)
        dz1 = da1 * (z1 > 0)
        dW1 = X.T.dot(dz1) / len(X)
        db1 = np.mean(dz1, axis=0, keepdims=True)
        
        # Update
        W2_temp -= lr * dW2
        b2_temp -= lr * db2
        W1_temp -= lr * dW1
        b1_temp -= lr * db1
    
    axes[idx].plot(temp_losses, color=colors[idx], linewidth=2)
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Loss')
    axes[idx].set_title(f'Learning Rate = {lr}', fontsize=12)
    axes[idx].grid(True, alpha=0.3)
    
    final_acc = np.mean((a2 > 0.5).astype(int).flatten() == y)
    axes[idx].text(100, max(temp_losses) * 0.8, f'Final Acc: {final_acc:.2%}', fontsize=10)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- LR = 0.001: Very slow, might not converge")
print("- LR = 0.1: Good balance, converges nicely")
print("- LR = 1.0: Too fast, might oscillate or diverge")

---
## 10. Experiment: Change Activation Function

Let's compare ReLU vs Sigmoid activation in the hidden layer.

In [ ]:
# Train with ReLU
np.random.seed(42)
W1_relu = np.random.randn(input_size, hidden_size) * 0.01
b1_relu = np.zeros((1, hidden_size))
W2_relu = np.random.randn(hidden_size, output_size) * 0.01
b2_relu = np.zeros((1, output_size))

losses_relu = []
for epoch in range(200):
    z1 = X.dot(W1_relu) + b1_relu
    a1 = relu(z1)
    z2 = a1.dot(W2_relu) + b2_relu
    a2 = sigmoid(z2)
    loss = -np.mean(y.reshape(-1, 1) * np.log(a2 + 1e-8) + 
                   (1 - y.reshape(-1, 1)) * np.log(1 - a2 + 1e-8))
    losses_relu.append(loss)
    dz2 = a2 - y.reshape(-1, 1)
    dW2 = a1.T.dot(dz2) / len(X)
    db2 = np.mean(dz2, axis=0, keepdims=True)
    da1 = dz2.dot(W2_relu.T)
    dz1 = da1 * (z1 > 0)
    dW1 = X.T.dot(dz1) / len(X)
    db1 = np.mean(dz1, axis=0, keepdims=True)
    W2_relu -= 0.1 * dW2
    b2_relu -= 0.1 * db2
    W1_relu -= 0.1 * dW1
    b1_relu -= 0.1 * db1

# Train with Sigmoid
np.random.seed(42)
W1_sig = np.random.randn(input_size, hidden_size) * 0.01
b1_sig = np.zeros((1, hidden_size))
W2_sig = np.random.randn(hidden_size, output_size) * 0.01
b2_sig = np.zeros((1, output_size))

losses_sig = []
for epoch in range(200):
    z1 = X.dot(W1_sig) + b1_sig
    a1 = sigmoid(z1)  # Sigmoid instead of ReLU
    z2 = a1.dot(W2_sig) + b2_sig
    a2 = sigmoid(z2)
    loss = -np.mean(y.reshape(-1, 1) * np.log(a2 + 1e-8) + 
                   (1 - y.reshape(-1, 1)) * np.log(1 - a2 + 1e-8))
    losses_sig.append(loss)
    dz2 = a2 - y.reshape(-1, 1)
    dW2 = a1.T.dot(dz2) / len(X)
    db2 = np.mean(dz2, axis=0, keepdims=True)
    da1 = dz2.dot(W2_sig.T)
    dz1 = da1 * a1 * (1 - a1)  # Sigmoid derivative
    dW1 = X.T.dot(dz1) / len(X)
    db1 = np.mean(dz1, axis=0, keepdims=True)
    W2_sig -= 0.1 * dW2
    b2_sig -= 0.1 * db2
    W1_sig -= 0.1 * dW1
    b1_sig -= 0.1 * db1

# Compare
plt.figure(figsize=(10, 5))
plt.plot(losses_relu, 'r-', linewidth=2, label='ReLU (Hidden Layer)')
plt.plot(losses_sig, 'b-', linewidth=2, label='Sigmoid (Hidden Layer)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ReLU vs Sigmoid: Training Loss', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nObservations:")
print("- ReLU typically converges faster")
print("- Sigmoid can suffer from vanishing gradients")
print("- ReLU is the most popular choice for hidden layers")

---
## 11. Summary

### What We Built:
1. A **single neuron** that computes weighted sum + activation
2. A **neural network** with input, hidden, and output layers
3. A **training loop** that learns through forward/backward passes

### What We Explored:
- How **activation functions** affect learning
- How **learning rate** affects convergence speed
- How the network learns a **decision boundary**

### Key Takeaways:
1. Neural networks learn by adjusting weights through **gradient descent**
2. **Forward propagation** flows data through the network
3. **Loss functions** measure how wrong predictions are
4. **Activation functions** enable learning of complex patterns
5. The **training loop** repeats until the network gets good at its task